> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 01. 이미지와 비디오 읽기·저장 — 55분

**예상 시간: 55분**

이 노트북은 영상 처리를 처음 배우는 비전공 학부생을 기준으로 합니다. 설명을 외우기보다 **입력 → 배열 → 변환 → 저장 → 재검증** 순서를 따라가세요.

목표는 `imread`/`imwrite`의 성공을 확인하고, 영상 frame 반복과 `release`가 필요한 이유를 설명하는 것입니다.

필요한 실행·저장 규칙은 이 노트북에서 다시 제공합니다. [06.OpenCV README](README.md)와 [용어 찾아보기](GLOSSARY.md)는 추가 참고 자료입니다. TODO는 JupyterLab 파일 편집기에서 저장하고, 미완성 검증이 실패하는 것은 정상입니다. 참고 정답 셀은 강사가 안내하는 시점에 펼칩니다.

수업 경로: 환경 5분 → 이미지 배열 10분 → TODO 20분 → 영상 15분 → 점검 5분.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

모델에 이미지를 넣기 전에 파일이 실제로 읽혔고 원하는 배열로 바뀌었는지 확인해야 합니다. 이 실습은 눈으로 보는 영상과 프로그램이 다루는 높이·너비·채널·dtype를 연결하며 저장 이후 다시 읽어 결과를 검증합니다.

### 전체 과정에서 내 코드가 하는 일

```text
파일
  ↓
BGR 배열
  ↓
학생 흑백 변환
  ↓
PNG 저장
  ↓
재열기·숫자 검사
  ↓
원본과 결과 나란히 보기
```

읽기 실패와 쓰기 성공 여부를 검사하는 TODO를 구현합니다. 영상에서는 프레임 읽기 성공을 매번 확인하고 capture/writer를 종료 시 반환합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 검은 화면이면 파일 읽기가 실패한 것인가요?

**설명:** 꼭 그렇지는 않습니다. None인지, shape·dtype·픽셀 범위가 무엇인지 먼저 확인합니다. 정상 검정 이미지와 읽기 실패는 다릅니다. 원본과 다시 읽은 흑백을 나란히 보고 저장 경로도 확인합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**OpenCV 준비:** Ollama·ROS node는 필요 없습니다. 첫 Python 셀의 `prepare_assets()`가 합성 입력을 준비하고 검증합니다. 입력은 `Data/images/06.OpenCV/sample_scene.png`와 `Data/videos/06.OpenCV/sample_motion.avi`, 출력은 `Data/outputs/06.OpenCV`입니다. 01 노트북을 먼저 실행해 만든 결과가 없어도 시작할 수 있습니다. GUI·USB 카메라는 로컬 컨테이너에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[02.Python](../02.Python/README.md)에서 배운 함수 반환값·조건문·반복문을 사용합니다. 이 실습에 필요한 내용은 `a,b = 함수()`로 두 반환값 받기, `if value is None:`으로 실패 검사하기, `try: ... finally: capture.release()`로 성공·오류 모두에서 장치를 해제하기입니다. 아래에 이미지 배열의 축과 저장 절차를 다시 설명하며 이전 Python 산출물을 읽지 않습니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **수업 핵심 TODO와 이미지·영상 입출력 경로** → [3일차 슬라이드 8쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s004) · [배포 PDF 8쪽](../슬라이드/3일차.pdf#page=8)에서 16쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 원본을 보존하고 변환·영상 저장 결과를 다시 엽니다.
- **단원 마지막 연결 질문:** 같은 작업대 대상의 밝기 또는 크기만 바꿔 mask가 달라지는지 비교합니다. 색 임계값으로 충분한 조건과 어려운 조건을 한 가지씩 적습니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib, json, subprocess, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '06.OpenCV/common').is_dir())
COURSE = ROOT / '06.OpenCV'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
import cv2, numpy as np
from common.vision import SAMPLE_IMAGE, SAMPLE_VIDEO, OUTPUT_DIR, load_image, prepare_assets, probe_video, save_image
from common.contracts import check
asset_report = prepare_assets()
print('Python:', sys.executable)
print('OpenCV:', cv2.__version__, 'NumPy:', np.__version__)
print(json.dumps(asset_report, ensure_ascii=False, indent=2))


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. 파일은 이미지 배열 자체가 아닙니다

PNG 파일은 픽셀을 압축하고 크기·색 형식 같은 정보를 함께 저장한 byte 묶음입니다. `cv2.imread`는 파일을 **decode** 해 `numpy.ndarray`로 만듭니다. 기본 color mode 결과는 `[H,W,3]`의 `uint8` BGR 배열입니다.

`image.shape == (360,640,3)`을 다음처럼 읽습니다.

| 축 | 값 | 뜻 |
|---|---:|---|
| 0 | 360 | 위에서 아래로 놓인 행, height |
| 1 | 640 | 왼쪽에서 오른쪽으로 놓인 열, width |
| 2 | 3 | B, G, R 세 channel |

파일 경로가 틀리거나 decode할 수 없으면 `imread`가 예외 대신 `None`을 돌려줄 수 있습니다. 따라서 shape를 쓰기 전에 `None` 검사를 해야 오류 원인이 입력 경로임을 바로 알 수 있습니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

### 실행 전 예측

1. 장면의 빨간 사각형 안 픽셀은 B, G, R 중 어느 채널이 가장 클까요?
2. grayscale로 바꾸면 channel 축은 남을까요?
3. `image[100,80]`의 두 위치 숫자는 `(x,y)`일까요, `(y,x)`일까요?


In [ ]:
# 구분: 수업 핵심 셀
image = load_image()
print('type:', type(image).__name__)
print('shape:', image.shape, '= (height, width, BGR channels)')
print('dtype/range:', image.dtype, int(image.min()), int(image.max()))
print('red rectangle pixel BGR:', image[100, 80].tolist())
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
print('gray shape:', gray.shape, 'channel 축이 사라졌습니다.')
assert image.shape == (360, 640, 3) and gray.shape == (360, 640)


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

`cv2.imwrite`는 성공 여부를 `True/False`로 반환합니다. 폴더가 없거나 권한·codec 문제가 있어도 반환값을 무시하면 뒤 단계에서야 문제를 발견합니다. 실습은 **쓰기→다시 읽기** 까지 한 작업으로 봅니다. PNG는 무손실이므로 grayscale 픽셀을 정확히 비교할 수 있습니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 수업 핵심 TODO — [io_task.py](exercises/io_task.py)

수정 파일: [exercises/io_task.py](exercises/io_task.py)

BGR 입력을 읽고 grayscale PNG로 저장한 뒤 다시 읽어 반환합니다. 없는 입력은 `FileNotFoundError`, 저장 실패는 `RuntimeError`로 원인을 분리하세요.

JupyterLab 파일 편집기에서 TODO를 구현하고 **저장** 한 뒤 아래 검증을 실행합니다. 검증은 저장된 모듈을 새 이름으로 다시 읽으므로 일반적으로 kernel 재시작은 필요 없습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `load_and_save_gray(input_path, output_path)`는 원본 이미지 경로와 저장할 PNG 경로를 받아 `(원본 BGR 배열, 저장 후 다시 읽은 gray 배열)`을 반환합니다. H×W 이미지라면 shape는 `(H,W,3)`과 `(H,W)`, dtype은 uint8입니다. 반환값뿐 아니라 output_path에 파일이 만들어집니다.
**작성 흐름 힌트:** 필요한 `cv2` import → IMREAD_COLOR로 읽고 None 검사 → BGR2GRAY → 출력 부모 폴더 생성 → imwrite의 bool 확인 → IMREAD_GRAYSCALE로 다시 읽고 None 확인 → 두 배열 반환. 입력과 출력은 서로 다른 경로를 사용하세요.
**설명할 지점:** “화면에 gray를 표시하는 것과 저장 파일을 검증하는 것은 다릅니다. 다시 읽어야 실제 저장 결과를 확인할 수 있습니다.”
**오류·통과 기준:** 입력을 못 읽으면 `FileNotFoundError`; `imwrite=False` 또는 저장 결과 재읽기 실패면 `RuntimeError`입니다. `cv2`는 starter에 없으므로 직접 import합니다. `output_path.parent.mkdir(parents=True, exist_ok=True)`가 중첩 폴더를 준비합니다. 검증은 저장된 PNG를 다시 읽은 값이 `cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)`와 픽셀까지 같은지 확인합니다. `check('io_task','exercise')`가 PASS여야 합니다. 검사 임시 파일은 지워지므로 아래 별도 실행·저장 셀로 자신의 gray 파일을 남깁니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
print((COURSE/'exercises/io_task.py').read_text())


In [ ]:
# 구분: 수업 핵심 셀
check('io_task', 'exercise')
print('learner_verification=PASS')


### 내 TODO로 결과 저장하기

검증용 PNG는 임시 폴더에서 검사 후 지워집니다. 검증을 통과한 **내 함수** 를 다시 읽어 `Data/outputs/06.OpenCV/01_gray.png`에 저장하고, 저장 후 다시 읽은 배열의 크기를 확인합니다. 재실행하면 이 출력 파일이 갱신됩니다.

In [ ]:
# 구분: 수업 핵심 셀
from common.contracts import _load
io_operation = _load('io_task', 'exercise').load_and_save_gray
original_bgr, decoded_gray = io_operation(SAMPLE_IMAGE, OUTPUT_DIR / '01_gray.png')
assert np.array_equal(decoded_gray, cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY))
print('saved:', OUTPUT_DIR / '01_gray.png', 'shape:', decoded_gray.shape)
# 저장·검증한 배열을 PNG로 표시합니다. 카메라나 모델을 새로 실행하지 않습니다.
import base64
import html
from IPython.display import HTML, display

def show_image_comparison(images):
    cards = []
    for pixels, title in images:
        ok, encoded = cv2.imencode('.png', pixels)
        assert ok, '비교 영상의 PNG 인코딩 실패'
        uri = 'data:image/png;base64,' + base64.b64encode(encoded.tobytes()).decode('ascii')
        cards.append(f'<figure style="margin:8px;flex:1;min-width:220px">'
                     f'<img src="{uri}" style="width:100%;height:280px;object-fit:contain" '
                     f'alt="{html.escape(title)}"><figcaption>{html.escape(title)}</figcaption></figure>')
    display(HTML('<div style="display:flex;flex-wrap:wrap">' + ''.join(cards) + '</div>'))

show_image_comparison([(original_bgr, '원본'), (decoded_gray, '저장 후 다시 읽은 흑백')])


> **구분: 수업 후 복습·자습** — 핵심 완료 뒤 원리와 확장을 확인합니다.

### 참고 정답

강사가 안내하는 시점에 정답과 해설을 비교합니다. 정답은 별도 파일이며 exercise를 덮어쓰지 않습니다. 구현이 한 글자씩 같을 필요는 없고 입력·출력·오류 처리 계약이 같으면 됩니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE/'solutions/io_task.py').read_text())


### 정답 해설 — 입력부터 결과까지

**읽기→변환→저장→재열기:** imread 결과가 None이면 뒤의 색 변환을 하지 않고 파일 오류를 냅니다. 성공한 BGR 3채널을 GRAY 1채널로 바꾼 뒤 출력 폴더를 만들고 imwrite의 성공 여부를 확인합니다. 마지막에는 저장 파일을 다시 읽어 반환합니다.

640×480 컬러 입력의 배열 모양은 `(480,640,3)`, 흑백은 `(480,640)`입니다. 파일명이 생겼다는 사실만으로 정상 저장을 입증하지 않으므로 재열기가 필요합니다. imshow는 창 표시, imwrite는 파일 저장이며 둘 중 하나가 다른 하나를 자동으로 수행하지 않습니다.

**확인 질문:** 원격 복습에서 창이 없어도 저장된 PNG를 Jupyter에서 열면 어떤 부분까지 확인할 수 있나요? 원본 파일과 출력 파일은 다른 경로를 사용하세요.

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/io_task.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""01_image_video_io 참고 정답."""

from pathlib import Path
import cv2
import numpy as np


def load_and_save_gray(input_path: Path, output_path: Path) -> tuple[np.ndarray, np.ndarray]:
    input_path, output_path = Path(input_path), Path(output_path)
    image = cv2.imread(str(input_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"이미지를 읽지 못했습니다: {input_path}")
    # 읽기 실패를 확인한 뒤에만 색 변환에 배열을 전달합니다.
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(output_path), gray):
        raise RuntimeError(f"이미지를 저장하지 못했습니다: {output_path}")
    # 메모리의 gray가 아니라 실제 디스크에 저장된 결과를 다시 검증합니다.
    decoded = cv2.imread(str(output_path), cv2.IMREAD_GRAYSCALE)
    if decoded is None:
        raise RuntimeError(f"저장 결과를 다시 읽지 못했습니다: {output_path}")
    return image, decoded
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
check('io_task', 'solution')
print('solution_verification=PASS')


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 2. 비디오는 시간 순서의 frame입니다

<!-- procedure:lesson-21-ff96d2:start -->
![영상을 읽고 결과를 저장하기](../교재그림/절차/lesson-21.svg)

읽기 실패·EOF·중단이면 반복을 끝내고 자원을 반환합니다. 파일 존재와 정상 재생을 함께 확인합니다.
<!-- procedure:lesson-21-ff96d2:end -->


영상은 frame 이미지, FPS와 codec 정보를 container 파일에 담습니다. `VideoCapture.read()`는 `(ok, frame)`을 반환합니다. `ok=False`는 파일 끝, 장치 끊김 또는 decode 실패일 수 있으므로 `frame`만 검사해서는 안 됩니다.

자원 수명은 `열기 → isOpened 확인 → 반복 read → finally에서 release`입니다. USB 카메라는 다른 프로세스와 공유되지 않을 수 있어 해제가 특히 중요합니다.


In [ ]:
# 구분: 수업 핵심 셀
info = probe_video()
capture = cv2.VideoCapture(str(SAMPLE_VIDEO))
assert capture.isOpened()
frames = []
try:
    for index in range(3):
        ok, frame = capture.read()
        assert ok and frame is not None
        frames.append(frame)
        print(index, frame.shape, frame.dtype)
finally:
    capture.release()
print('metadata:', info, 'read frames:', len(frames))


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

### 원격에서도 같은 처리를 확인하는 방법

GUI 창의 존재는 처리 성공의 증거가 아닙니다. 다음 프로그램은 고정 녹화 입력 36 frame에 번호를 그리고 AVI와 JSON을 Data에 저장한 뒤 다시 엽니다. `--headless`는 `imshow`만 생략합니다.


In [ ]:
# 구분: 수업 핵심 셀
command = [sys.executable, str(COURSE/'scripts/video_pipeline.py'),
           '--source', 'sample', '--max-frames', '36', '--headless']
run = subprocess.run(command, text=True, capture_output=True, timeout=60)
print(run.stdout)
if run.stderr: print(run.stderr)
run.check_returncode()
summary = json.loads((OUTPUT_DIR/'video_pipeline.json').read_text())
assert summary['read'] == summary['written'] == 36


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 결과 해석과 완료 질문

- `shape`가 맞다는 사실과 이미지 내용이 맞다는 사실은 다릅니다. 이 TODO는 예상 grayscale 픽셀까지 비교합니다.
- 파일 frame 수는 codec에 따라 메타데이터가 0일 수 있어 첫 frame decode와 처리 중 직접 센 수를 함께 사용합니다.
- USB camera 실습은 로컬에서 `--source camera`로 한 번 실행하고 두 수강생이 같은 화면을 함께 봅니다.

**말로 답하기:** `read()`가 실패했을 때 무한 반복하지 않는 조건과, 정상·오류 어느 경로에서도 `release()`가 실행되게 하는 Python 구문은 무엇인가요?




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** PNG 재열기와 36프레임 영상 처리·자원 반환 확인

[3일차 슬라이드 8쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s004) · [배포 PDF 8쪽](../슬라이드/3일차.pdf#page=8)로 돌아가 **8–16쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습** — 핵심 완료 뒤 원리와 확장을 확인합니다.

## 자습 — JPEG, backend와 codec

JPEG는 손실 압축이라 저장 결과 픽셀이 조금 달라질 수 있습니다. PNG와 같은 `array_equal` 검사를 그대로 적용하지 않습니다. 카메라 `VideoCapture(0)`의 실제 backend는 V4L2/GStreamer 등 build와 입력 문자열에 따라 달라질 수 있습니다. 영상 확장자는 container, `MJPG`·`H264`는 codec이므로 한 개념으로 부르지 않습니다.
